<a href="https://colab.research.google.com/github/yojith712/python/blob/main/ISEF_V4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
#  Genetic Disease Symptom Checker  v2  (school project demo)
#  Paste this whole thing into one Colab cell and run it.
#  NOT a medical tool - always see a doctor / genetic counselor.
#
#  What it does:
#   1. Reads your symptoms and turns them into standard HPO terms
#   2. Compares them to a database of genetic diseases
#   3. If it's not sure, it ASKS YOU follow-up questions
#   4. Shows the best matches, WHERE in the DNA each one is,
#      HOW the disease happens, and HOW the program decided
# ==========================================================
import math
import re
import difflib

# ----------------------------------------------------------
# STEP 1: Symptom vocabulary -> standard HPO terms
#   (Verify IDs at https://hpo.jax.org before using real data.)
# ----------------------------------------------------------
SYMPTOMS = {
    # --- general / development ---
    "HP:0001249": ("Intellectual disability", ["intellectual disability", "learning disability", "slow learning", "mental disability"]),
    "HP:0001263": ("Developmental delay", ["developmental delay", "delayed development", "slow development", "development delay"]),
    "HP:0001328": ("Learning difficulties", ["learning difficulties", "learning problems", "trouble reading", "dyslexia", "trouble in school"]),
    "HP:0000750": ("Delayed speech", ["delayed speech", "late talking", "talked late", "speech delay", "slow to talk", "not talking"]),
    "HP:0012378": ("Fatigue", ["fatigue", "tired", "tiredness", "no energy", "exhausted"]),
    "HP:0004322": ("Short stature", ["short stature", "very short", "short height", "small for age", "not growing", "shorter than others"]),
    "HP:0000098": ("Tall stature", ["very tall", "tall stature", "tall"]),
    "HP:0001508": ("Failure to thrive", ["poor weight gain", "not gaining weight", "failure to thrive", "underweight"]),
    # --- face / body features ---
    "HP:0000582": ("Upward-slanting eyes", ["slanted eyes", "upward slanting eyes", "upslanting eyes", "almond shaped eyes", "almond eyes", "eyes slant up"]),
    "HP:0012368": ("Flat face / flat nose bridge", ["flat face", "flat facial profile", "flat nose", "flat nasal bridge", "flat nose bridge"]),
    "HP:0000954": ("Single crease across palm", ["single palm crease", "single crease", "one crease on palm", "one line on palm", "palmar crease"]),
    "HP:0010808": ("Tongue sticks out", ["tongue sticks out", "protruding tongue", "big tongue", "large tongue"]),
    "HP:0008551": ("Small ears", ["small ears", "tiny ears"]),
    "HP:0000400": ("Large ears", ["large ears", "big ears"]),
    "HP:0000276": ("Long face", ["long face"]),
    "HP:0000465": ("Webbed neck", ["webbed neck", "wide neck", "extra skin on neck"]),
    "HP:0001166": ("Long fingers", ["long fingers", "spider fingers", "long thin fingers"]),
    "HP:0000767": ("Sunken chest", ["sunken chest", "pectus", "caved in chest"]),
    "HP:0003707": ("Enlarged calf muscles", ["big calves", "enlarged calves", "large calves"]),
    # --- muscles / movement ---
    "HP:0001324": ("Muscle weakness", ["muscle weakness", "weak muscles", "weakness", "can't lift", "cant lift"]),
    "HP:0001252": ("Low muscle tone (floppy)", ["floppy", "low muscle tone", "hypotonia", "loose muscles"]),
    "HP:0002359": ("Frequent falls", ["falls a lot", "frequent falls", "falling", "falls often"]),
    "HP:0003551": ("Difficulty climbing stairs", ["climbing stairs", "trouble with stairs", "hard to climb stairs"]),
    "HP:0002515": ("Waddling walk", ["waddling", "waddle"]),
    "HP:0002194": ("Late walking", ["late walking", "walked late", "delayed walking", "slow to walk"]),
    "HP:0002072": ("Involuntary movements (chorea)", ["involuntary movements", "jerky movements", "uncontrolled movements", "chorea"]),
    "HP:0001337": ("Tremor", ["tremor", "shaking", "shaky hands"]),
    "HP:0002376": ("Losing skills already learned", ["losing skills", "loss of motor skills", "regression", "forgot how to"]),
    "HP:0001382": ("Very flexible joints", ["flexible joints", "double jointed", "loose joints", "hypermobile"]),
    # --- brain / mood ---
    "HP:0001250": ("Seizures", ["seizure", "seizures", "convulsion", "convulsions"]),
    "HP:0002354": ("Memory problems", ["memory loss", "memory problems", "forgetful", "forgetting"]),
    "HP:0000716": ("Depression / mood changes", ["depression", "depressed", "mood changes", "mood swings"]),
    "HP:0000739": ("Anxiety", ["anxiety", "anxious", "nervous"]),
    "HP:0000752": ("Hyperactivity", ["hyperactive", "hyperactivity", "can't sit still", "cant sit still"]),
    "HP:0002267": ("Exaggerated startle", ["startle", "jumpy", "easily startled"]),
    # --- lungs / gut ---
    "HP:0012735": ("Cough", ["cough", "coughing", "chronic cough"]),
    "HP:0002205": ("Recurrent lung infections", ["lung infections", "chest infections", "pneumonia", "respiratory infections"]),
    "HP:0002719": ("Recurrent infections", ["frequent infections", "get sick often", "recurrent infections"]),
    "HP:0002094": ("Shortness of breath", ["shortness of breath", "short of breath", "breathing problems", "hard to breathe", "trouble breathing"]),
    "HP:0002015": ("Trouble swallowing", ["trouble swallowing", "hard to swallow", "difficulty swallowing", "choking"]),
    "HP:0012236": ("Salty sweat", ["salty skin", "salty sweat", "tastes salty"]),
    "HP:0002570": ("Greasy stools", ["greasy stool", "greasy poop", "oily stool", "fatty stool"]),
    "HP:0002027": ("Abdominal pain", ["stomach pain", "belly pain", "abdominal pain", "stomach ache"]),
    "HP:0002240": ("Enlarged liver", ["enlarged liver", "liver problems", "big liver"]),
    "HP:0001744": ("Enlarged spleen", ["enlarged spleen", "big spleen", "spleen"]),
    # --- blood ---
    "HP:0001903": ("Anemia", ["anemia", "anaemia", "low blood", "pale"]),
    "HP:0000952": ("Jaundice (yellow skin/eyes)", ["jaundice", "yellow skin", "yellow eyes"]),
    "HP:0100338": ("Swollen hands/feet (dactylitis)", ["swollen hands", "swollen feet", "swelling in hands", "swelling in feet"]),
    "HP:0000978": ("Easy bruising", ["bruise easily", "easy bruising", "bruising", "bruises"]),
    "HP:0001892": ("Abnormal bleeding", ["bleeding", "bleed a lot", "won't stop bleeding", "wont stop bleeding"]),
    "HP:0000421": ("Nosebleeds", ["nosebleed", "nosebleeds", "nose bleeds"]),
    "HP:0005261": ("Bleeding into joints", ["swollen joints", "bleeding into joints", "joint swelling"]),
    "HP:0002829": ("Joint pain", ["joint pain", "joints hurt", "achy joints", "pain in joints"]),
    # --- heart ---
    "HP:0001627": ("Heart defect", ["heart defect", "heart problem", "heart problems", "hole in heart", "hole in the heart", "heart murmur"]),
    "HP:0002616": ("Enlarged aorta", ["aorta", "aortic", "enlarged aorta"]),
    # --- eyes ---
    "HP:0001083": ("Dislocated eye lens", ["dislocated lens", "lens dislocation"]),
    "HP:0000572": ("Vision loss", ["vision loss", "losing vision", "blind", "can't see", "cant see", "blurry vision"]),
    "HP:0010729": ("Cherry-red spot in eye", ["cherry red spot", "cherry-red spot"]),
    "HP:0200032": ("Brown ring around iris", ["kayser", "brown ring", "ring around iris"]),
    # --- skin ---
    "HP:0000964": ("Eczema", ["eczema", "itchy rash", "rash"]),
    "HP:0001010": ("Light skin / hair", ["light skin", "very pale skin", "light hair"]),
    "HP:0000953": ("Darkened skin", ["dark skin patches", "bronze skin", "skin darkening"]),
    "HP:0001004": ("Puffy hands/feet (lymphedema)", ["lymphedema", "puffy hands", "puffy feet"]),
    # --- puberty / fertility ---
    "HP:0000823": ("Delayed puberty", ["delayed puberty", "late puberty", "no puberty", "puberty late"]),
    "HP:0000789": ("Infertility", ["infertility", "infertile", "can't have kids", "cant have kids"]),
    "HP:0000771": ("Breast growth in males", ["gynecomastia", "breast growth in boys", "enlarged breasts"]),
}

# ----------------------------------------------------------
# STEP 2: Disease database
#   location : cytogenetic band (chromosome + arm + band)
#   kind     : "gene" (one gene mutated) or "chromosome" (whole/part chromosome)
#   cause    : what goes wrong in the DNA
#   how      : how that DNA change leads to the symptoms
# ----------------------------------------------------------
DISEASES = [
    dict(name="Down syndrome", kind="chromosome", gene=None,
         location="21", dna="An entire EXTRA copy of chromosome 21 (trisomy 21) - 47 chromosomes instead of 46",
         inheritance="Usually NOT inherited - happens by chance when an egg or sperm forms (nondisjunction)",
         how="Having 3 copies of chromosome 21 means about 1.5x the normal amount of every gene on it. "
             "That extra 'dose' changes how the brain, face, heart and muscles develop before birth.",
         hpos=["HP:0001249", "HP:0001263", "HP:0000582", "HP:0012368", "HP:0000954", "HP:0001252",
               "HP:0004322", "HP:0001627", "HP:0010808", "HP:0000750", "HP:0008551"]),
    dict(name="Turner syndrome", kind="chromosome", gene=None,
         location="X", dna="One X chromosome is missing (45,X) - affects females",
         inheritance="Usually NOT inherited - happens by chance",
         how="Some genes on the X chromosome need 2 working copies. The SHOX gene missing a copy causes short "
             "height, and the ovaries don't develop normally, so puberty is delayed.",
         hpos=["HP:0004322", "HP:0000465", "HP:0000823", "HP:0001004", "HP:0001627", "HP:0000789", "HP:0001328"]),
    dict(name="Klinefelter syndrome", kind="chromosome", gene=None,
         location="X", dna="An EXTRA X chromosome in a male (47,XXY)",
         inheritance="Usually NOT inherited - happens by chance",
         how="The extra X lowers testosterone production, which affects puberty, body shape and fertility.",
         hpos=["HP:0000098", "HP:0000823", "HP:0000771", "HP:0000789", "HP:0001328", "HP:0000750"]),
    dict(name="Cystic fibrosis", kind="gene", gene="CFTR", location="7q31.2",
         dna="Mutation in the CFTR gene (most common: deltaF508, 3 DNA letters deleted)",
         inheritance="Autosomal recessive - needs a broken copy from BOTH parents",
         how="CFTR makes a channel that moves salt (chloride) in and out of cells. When it's broken, mucus becomes "
             "thick and sticky, clogging the lungs and pancreas, and sweat comes out extra salty.",
         hpos=["HP:0012735", "HP:0002205", "HP:0012236", "HP:0001508", "HP:0002570", "HP:0002094"]),
    dict(name="Sickle cell disease", kind="gene", gene="HBB", location="11p15.4",
         dna="One DNA letter changed (A->T) in the HBB gene, making 'hemoglobin S'",
         inheritance="Autosomal recessive - needs a broken copy from BOTH parents",
         how="HBB makes part of hemoglobin, which carries oxygen. Hemoglobin S makes red blood cells bend into a "
             "sickle shape. They get stuck in blood vessels (pain, swelling) and break apart early (anemia, jaundice).",
         hpos=["HP:0001903", "HP:0012378", "HP:0000952", "HP:0100338", "HP:0002719", "HP:0001744"]),
    dict(name="Duchenne muscular dystrophy", kind="gene", gene="DMD", location="Xp21.2",
         dna="Large deletion in the DMD (dystrophin) gene - one of the biggest genes in the body",
         inheritance="X-linked recessive - mostly affects boys",
         how="Dystrophin acts like a shock absorber for muscle cells. Without it, muscles get damaged every time "
             "they're used and are replaced by fat and scar tissue (that's why calves look big but are weak).",
         hpos=["HP:0001324", "HP:0002359", "HP:0003551", "HP:0003707", "HP:0002515", "HP:0002194"]),
    dict(name="Spinal muscular atrophy", kind="gene", gene="SMN1", location="5q13.2",
         dna="Both copies of the SMN1 gene deleted",
         inheritance="Autosomal recessive - needs a broken copy from BOTH parents",
         how="SMN1 keeps motor nerve cells in the spinal cord alive. Without it, those nerves die, so muscles "
             "stop getting signals and become weak and floppy.",
         hpos=["HP:0001324", "HP:0001252", "HP:0002094", "HP:0002015", "HP:0002194"]),
    dict(name="Huntington disease", kind="gene", gene="HTT", location="4p16.3",
         dna="A 'CAG' DNA repeat in the HTT gene is too long (36+ repeats instead of ~20)",
         inheritance="Autosomal dominant - only ONE broken copy needed",
         how="The extra-long repeat makes a sticky huntingtin protein that slowly kills brain cells that control "
             "movement, mood and thinking.",
         hpos=["HP:0002072", "HP:0000716", "HP:0002354", "HP:0002015", "HP:0001337"]),
    dict(name="Marfan syndrome", kind="gene", gene="FBN1", location="15q21.1",
         dna="Mutation in the FBN1 gene",
         inheritance="Autosomal dominant - only ONE broken copy needed",
         how="FBN1 makes fibrillin-1, a protein that makes connective tissue strong and stretchy. Weak connective "
             "tissue lets bones grow too long and makes joints, eye lenses and the aorta loose or stretched.",
         hpos=["HP:0000098", "HP:0001166", "HP:0001382", "HP:0001083", "HP:0000767", "HP:0002616", "HP:0001627"]),
    dict(name="Hemophilia A", kind="gene", gene="F8", location="Xq28",
         dna="Mutation in the F8 gene (often a flipped section of DNA)",
         inheritance="X-linked recessive - mostly affects boys",
         how="F8 makes clotting factor VIII. Without enough of it, blood can't form clots properly, so bleeding "
             "lasts longer and blood leaks into joints.",
         hpos=["HP:0000978", "HP:0001892", "HP:0000421", "HP:0005261", "HP:0002829"]),
    dict(name="Phenylketonuria (PKU)", kind="gene", gene="PAH", location="12q23.2",
         dna="Mutation in the PAH gene",
         inheritance="Autosomal recessive - needs a broken copy from BOTH parents",
         how="PAH makes an enzyme that breaks down phenylalanine (found in protein foods). It builds up to toxic "
             "levels and harms the developing brain. Less melanin is made, so skin and hair are lighter.",
         hpos=["HP:0001249", "HP:0001250", "HP:0000964", "HP:0000752", "HP:0001010"]),
    dict(name="Tay-Sachs disease", kind="gene", gene="HEXA", location="15q23",
         dna="Mutation in the HEXA gene",
         inheritance="Autosomal recessive - needs a broken copy from BOTH parents",
         how="HEXA makes an enzyme that clears a fatty substance (GM2) from nerve cells. Without it, the fat builds "
             "up and destroys nerve cells in the brain, so the child loses skills, sight and control.",
         hpos=["HP:0002376", "HP:0001250", "HP:0000572", "HP:0010729", "HP:0002267", "HP:0001324"]),
    dict(name="Fragile X syndrome", kind="gene", gene="FMR1", location="Xq27.3",
         dna="A 'CGG' DNA repeat in the FMR1 gene is far too long (200+ repeats), switching the gene off",
         inheritance="X-linked - boys usually more affected",
         how="FMR1 makes a protein (FMRP) that brain cells need to form connections. Without it, learning, "
             "speech and behavior are affected.",
         hpos=["HP:0001249", "HP:0001263", "HP:0000400", "HP:0000276", "HP:0000739", "HP:0000752", "HP:0000750"]),
    dict(name="Wilson disease", kind="gene", gene="ATP7B", location="13q14.3",
         dna="Mutation in the ATP7B gene",
         inheritance="Autosomal recessive - needs a broken copy from BOTH parents",
         how="ATP7B helps the liver get rid of extra copper. Copper builds up and damages the liver, brain "
             "(tremor, mood) and eyes (brown ring around the iris).",
         hpos=["HP:0002240", "HP:0000952", "HP:0001337", "HP:0000716", "HP:0200032", "HP:0002027"]),
    dict(name="Hereditary hemochromatosis", kind="gene", gene="HFE", location="6p22.2",
         dna="Mutation in the HFE gene (most common: C282Y)",
         inheritance="Autosomal recessive - needs a broken copy from BOTH parents",
         how="HFE helps control how much iron the gut absorbs. The body absorbs too much iron, which gets stored "
             "in the liver, joints and skin and slowly damages them.",
         hpos=["HP:0012378", "HP:0002829", "HP:0002027", "HP:0000953", "HP:0002240"]),
]

# ----------------------------------------------------------
# Rarity weights: a symptom found in few diseases counts more
# ----------------------------------------------------------
def build_weights():
    counts = {}
    for d in DISEASES:
        for h in d["hpos"]:
            counts[h] = counts.get(h, 0) + 1
    n = len(DISEASES)
    return {h: math.log(n / c) + 1 for h, c in counts.items()}

WEIGHTS = build_weights()

# Longest phrases first so "short of breath" wins over shorter words
PHRASES = sorted(
    [(p, h) for h, (_, words) in SYMPTOMS.items() for p in words],
    key=lambda x: -len(x[0]),
)
PHRASE_TO_HPO = dict(PHRASES)

# ----------------------------------------------------------
# Explain a DNA location like "7q31.2" in plain words
# ----------------------------------------------------------
def explain_location(d):
    if d["kind"] == "chromosome":
        return f"Chromosome {d['location']} (the whole chromosome)"
    m = re.match(r"^(\d+|X|Y)([pq])(.+)$", d["location"])
    if not m:
        return d["location"]
    chrom, arm, band = m.groups()
    arm_word = "short arm (p)" if arm == "p" else "long arm (q)"
    chrom_word = f"{chrom} chromosome" if chrom in ("X", "Y") else f"chromosome {chrom}"
    return f"{d['gene']} gene on {chrom_word}, {arm_word}, band {band}  [{d['location']}]"

# ----------------------------------------------------------
# STEP 1 (code): turn what the user typed into HPO terms
# ----------------------------------------------------------
def text_to_hpo(text):
    text = " " + text.lower().replace("\n", " ") + " "
    found = {}
    for phrase, hpo in PHRASES:
        pattern = r"(?<![a-z])" + re.escape(phrase) + r"(?![a-z])"   # whole words only
        if re.search(pattern, text):
            if hpo not in found:
                found[hpo] = phrase
            text = re.sub(pattern, " , ", text)  # don't reuse the same words
    # Typo help: compare leftover pieces to known phrases
    all_phrases = [p for p, _ in PHRASES]
    for piece in re.split(r",| and ", text):
        piece = piece.strip(" .;")
        if len(piece) < 4:
            continue
        close = difflib.get_close_matches(piece, all_phrases, n=1, cutoff=0.8)
        if close:
            found.setdefault(PHRASE_TO_HPO[close[0]], close[0] + " (typo fixed)")
    return found

# ----------------------------------------------------------
# STEP 3: score every disease
#   coverage  = how much of the disease's symptom list you have
#   explained = how many of YOUR symptoms the disease explains
#   penalty   = symptoms you said you DON'T have lower the score
# ----------------------------------------------------------
def score_diseases(have, denied=frozenset(), flagged_genes=frozenset()):
    results = []
    user_total = sum(WEIGHTS.get(h, 1) for h in have)
    for d in DISEASES:
        hpos = d["hpos"]
        matched = [h for h in hpos if h in have]
        dna_hit = d["gene"] is not None and d["gene"] in flagged_genes
        if not matched and not dna_hit:
            continue
        total_w = sum(WEIGHTS[h] for h in hpos)
        w_matched = sum(WEIGHTS[h] for h in matched)
        w_denied = sum(WEIGHTS[h] for h in hpos if h in denied)
        coverage = w_matched / total_w
        explained = w_matched / user_total if user_total else 0
        score = 0.6 * coverage + 0.4 * explained - 0.5 * (w_denied / total_w)
        if dna_hit:
            score += 0.30  # STEP 5: DNA evidence boost
        score = max(0.0, min(1.0, score))
        results.append(dict(
            d=d, score=score, matched=matched,
            ruled_out=[h for h in hpos if h in denied],
            unknown=[h for h in hpos if h not in have and h not in denied],
            dna_hit=dna_hit,
        ))
    return sorted(results, key=lambda r: -r["score"])

def is_confident(results, have):
    if not results or len(have) < 3:
        return False
    top = results[0]["score"]
    second = results[1]["score"] if len(results) > 1 else 0
    return top >= 0.6 and (top - second) >= 0.2

# ----------------------------------------------------------
# NEW: ask follow-up questions when the program isn't sure.
#   Picks symptoms that best tell the top candidates apart.
# ----------------------------------------------------------
def ask_follow_ups(have, denied, results, flagged, max_rounds=3, per_round=4):
    asked = set(have) | set(denied)
    for rnd in range(max_rounds):
        if is_confident(results, have):
            break
        top = results[:3]
        if not top:
            break
        priority = {}
        for r in top:
            for h in r["unknown"]:
                if h in asked:
                    continue
                in_how_many = sum(1 for t in top if h in t["d"]["hpos"])
                # rare symptoms of high-scoring diseases that only ONE candidate has = most useful
                p = WEIGHTS[h] * (0.5 + r["score"]) / in_how_many
                priority[h] = max(priority.get(h, 0), p)
        questions = sorted(priority, key=lambda h: -priority[h])[:per_round]
        if not questions:
            break

        if rnd == 0:
            print("\nI'm not sure yet. I need more info to narrow it down.")
            print("Answer  y = yes,  n = no,  ? = not sure\n")
        else:
            print("\nA few more questions:\n")
        for h in questions:
            ans = input(f"  Do you have: {SYMPTOMS[h][0].lower()}?  ").strip().lower()
            asked.add(h)
            if ans.startswith("y"):
                have[h] = "yes (answered follow-up)"
            elif ans.startswith("n"):
                denied.add(h)
        results = score_diseases(set(have), denied, flagged)
    return results

# ----------------------------------------------------------
# STEP 5: look for flagged genes in your sample_variants table
# ----------------------------------------------------------
def genes_from_variants():
    df = globals().get("sample_variants")
    if df is None:
        return set()
    gene_col = next((c for c in df.columns
                     if c.lower() in ("gene", "gene_name", "genesymbol", "gene_symbol", "symbol")), None)
    if gene_col is None:
        print("(sample_variants found, but no 'gene' column - skipping DNA boost)")
        return set()
    return set(df[gene_col].dropna().astype(str).str.upper())

# ----------------------------------------------------------
# STEP 4: show results
# ----------------------------------------------------------
def label(score):
    if score >= 0.6: return "HIGH match"
    if score >= 0.35: return "MEDIUM match"
    return "LOW match"

def names(hpos):
    return ", ".join(SYMPTOMS[h][0] for h in hpos)

def show_results(have, denied, results, detailed=3, top_n=5):
    print("\n" + "=" * 62)
    print("SYMPTOMS USED")
    print("=" * 62)
    for h, phrase in have.items():
        print(f"  + {SYMPTOMS[h][0]}  ({h})  <- '{phrase}'")
    for h in denied:
        print(f"  - NOT: {SYMPTOMS[h][0]}")

    if not results:
        print("\nNo disease in the database matches those symptoms.")
    else:
        print("\n" + "=" * 62)
        print("RESULTS")
        print("=" * 62)
        for i, r in enumerate(results[:top_n], 1):
            d = r["d"]
            pct = round(r["score"] * 100)
            bar = "#" * (pct // 5) + "-" * (20 - pct // 5)
            print(f"\n{i}. {d['name'].upper()}")
            print(f"   Match score: {pct}%  [{bar}]  {label(r['score'])}")
            if i > detailed:
                print(f"   (matched: {names(r['matched']) or 'none'})")
                continue
            print(f"\n   WHERE IN THE DNA:")
            print(f"     {explain_location(d)}")
            print(f"     Change: {d['dna']}")
            print(f"     Inherited: {d['inheritance']}")
            print(f"\n   HOW THE DISEASE HAPPENS:")
            print(f"     {d['how']}")
            print(f"\n   HOW I DECIDED:")
            n_total = len(d["hpos"])
            print(f"     You have {len(r['matched'])} of its {n_total} main symptoms.")
            if r["matched"]:
                print(f"     Matches:     {names(r['matched'])}")
            if r["ruled_out"]:
                print(f"     You said no: {names(r['ruled_out'])}  (lowered the score)")
            if r["unknown"]:
                print(f"     Not checked: {names(r['unknown'])}")
            if r["dna_hit"]:
                print(f"     DNA data: a variant in {d['gene']} is in sample_variants (+30%)")
            print("   " + "-" * 59)

        if not is_confident(results, have):
            print("\n  NOTE: I'm still NOT confident - the top results are close or too")
            print("  few symptoms matched. A doctor and genetic test are needed.")

    print("\nIMPORTANT: The match score shows how well symptoms line up with")
    print("this small database - it is NOT the chance you have the disease.")
    print("Only a doctor and a genetic test can diagnose a genetic disease.")

# ----------------------------------------------------------
# Main program
# ----------------------------------------------------------
def run_checker():
    print("=" * 62)
    print("     GENETIC DISEASE SYMPTOM CHECKER  v2  (school project)")
    print("=" * 62)
    print("Describe the symptoms, separated by commas.")
    print("Example: slanted eyes, flat face, low muscle tone, heart defect")
    print("Type 'list' to see symptoms I know, or 'quit' to exit.\n")

    flagged = genes_from_variants()
    if flagged:
        print(f"(Using DNA data: {len(flagged)} gene(s) flagged in sample_variants)\n")

    while True:
        text = input("Symptoms: ").strip()
        if text.lower() in ("quit", "exit", "q"):
            print("Goodbye!")
            break
        if text.lower() == "list":
            for name, _ in sorted(SYMPTOMS.values()):
                print("  -", name)
            print()
            continue
        if not text:
            continue

        have = text_to_hpo(text)
        if not have:
            print("I didn't recognize any symptoms. Try different words or type 'list'.\n")
            continue

        denied = set()
        results = score_diseases(set(have), denied, flagged)

        print("\nFirst guess:")
        for r in results[:3]:
            print(f"  - {r['d']['name']}: {round(r['score'] * 100)}%")

        results = ask_follow_ups(have, denied, results, flagged)
        show_results(have, denied, results)

        print()
        if input("Check another set of symptoms? (y/n): ").strip().lower() != "y":
            print("Goodbye!")
            break

run_checker()

     GENETIC DISEASE SYMPTOM CHECKER  v2  (school project)
Describe the symptoms, separated by commas.
Example: slanted eyes, flat face, low muscle tone, heart defect
Type 'list' to see symptoms I know, or 'quit' to exit.

  - Abdominal pain
  - Abnormal bleeding
  - Anemia
  - Anxiety
  - Bleeding into joints
  - Breast growth in males
  - Brown ring around iris
  - Cherry-red spot in eye
  - Cough
  - Darkened skin
  - Delayed puberty
  - Delayed speech
  - Depression / mood changes
  - Developmental delay
  - Difficulty climbing stairs
  - Dislocated eye lens
  - Easy bruising
  - Eczema
  - Enlarged aorta
  - Enlarged calf muscles
  - Enlarged liver
  - Enlarged spleen
  - Exaggerated startle
  - Failure to thrive
  - Fatigue
  - Flat face / flat nose bridge
  - Frequent falls
  - Greasy stools
  - Heart defect
  - Hyperactivity
  - Infertility
  - Intellectual disability
  - Involuntary movements (chorea)
  - Jaundice (yellow skin/eyes)
  - Joint pain
  - Large ears
  - Late walki